# Lecture: Implementation of Policy Iteration

We want to implement policy iteration for the `frozen lake` environment provided by gymnasium. Gymnasium is a reinforcement learning (RL) environment library that provides a collection of pre-built environments for training and evaluating RL algorithms. It is a maintained fork of the original OpenAI Gym, which became unmaintained after OpenAI shifted focus.

The Environments follow a universal API, making it easy to test different RL algorithms on various tasks.

If you execute the code locally and have pygame installed as in the requirements.txt a window should pop up if you choose render_mode='human' showing the env and dynamics.

In [ ]:
!git clone https://github.com/Fjoelsak/RL.git
!cp RL/03_Dynamic_Programming/mdp_control.py ./

In [1]:
import gymnasium as gym

env = gym.make('FrozenLake-v1',
               desc = None,
               map_name = "4x4",
               is_slippery = False,
               render_mode = 'human')

This is an exemplary use of the provided environment in which the agent takes a randomly sampled action in each state.

In [ ]:
obs,_ = env.reset()

while True:
    env.render()
    action = env.action_space.sample()

    obs, reward, terminated, truncated, _ = env.step(action)

    if terminated or truncated:
        break

env.close()

If the window is not closing, you can manually quit pygame by executing the following cell.

In [ ]:
import pygame

pygame.quit()

# Excercise 1: Getting to know the environment

Go to the farama [foundation docs of the environment](https://gymnasium.farama.org/environments/toy_text/frozen_lake/) and determine how the state and action spaces are defined, how the reward function is implemented and how the condition for a termination of the episode is implemented. In addition, check what the `is_slippery` boolean is doing.

In [ ]:
env.observation_space

In [ ]:
env.action_space

# Excercise 2: Policy Iteration for the Frozen lake environment

Check the class `mdpControl` in `mdp_control.py` and implement the functions `policy_evaluation()` and `policy_iteration()`.

In order to get the transition probability matrix explicity you can use the unwrapped environment of the frozen lake env. With `env.unwrapped.P` you get for each state (0-15) for each action (0-3) the corresponding transition probability, next_state, reward and a done flag whether the episode is terminated. If the environment is stochastic, there will be several entries for each state and action pair. You can check by enabling the `is_slippery` flag and look at the `env.unwrappd.P` object.

In [ ]:
env.unwrapped.P

In [ ]:
import gymnasium as gym
import mdp_control as mdp_control

env = gym.make('FrozenLake-v1',
               desc = None,
               map_name = "4x4",
               is_slippery = False,
               render_mode = 'human')

mdp = mdp_control.mdpControl(env)
p, V, n_iter = mdp.policy_iteration()
print(f"Policy iteration converged in {n_iter} iterations")

# rendering of the agent acting in the env with your optimized policy
mdp.render_single(p)
env.close()

# Excercise 3: Value Iteration for the Frozen lake environment

Check the class `mdpControl` in `mdp_control.py` and implement the function `value_iteration()`.

In [ ]:
import mdp_control

env = gym.make('FrozenLake-v1',
               desc = None,
               map_name = "4x4",
               is_slippery = False,
               render_mode = 'human')

mdp = mdp_control.mdpControl(env)
p, V, n_iter = mdp.value_iteration()
print(f"Value iteration converged in {n_iter} iterations")

# rendering of the agent acting in the env with your optimized policy
mdp.render_single(p)
env.close()

# Excercise 4: Stochasticity

Test both methods (policy and value iteration) in the frozen lake environment with `is_slippery=False` and `is_slippery=True`. In the second environment, the dynamics of the world are stochastic.

- Check in the frozen lake docs of the farama foundation how the transition probabilites are affected by the `is_slippery` flag
- How does stochasticity affect the number of iterations required, and the resulting policy?


In [ ]:
import importlib
import mdp_control

results = {}

for slippery in [False, True]:
    env = gym.make('FrozenLake-v1', map_name="4x4", is_slippery=slippery)
    mdp = mdp_control.mdpControl(env)

    p_pi, V_pi, n_pi = mdp.policy_iteration()
    p_vi, V_vi, n_vi = mdp.value_iteration()

    results[slippery] = {
        "p_pi": p_pi, "V_pi": V_pi, "n_pi": n_pi,
        "p_vi": p_vi, "V_vi": V_vi, "n_vi": n_vi,
    }
    env.close()

# Iterations comparison
print("Iterations to convergence:")
print(f"{'':20} {'deterministic':>15} {'stochastic':>12}")
print(f"{'Policy Iteration':20} {results[False]['n_pi']:>15} {results[True]['n_pi']:>12}")
print(f"{'Value Iteration':20} {results[False]['n_vi']:>15} {results[True]['n_vi']:>12}")

# State-value comparison
print("\nState values (all 16 states):")
print(f"{'State':<8} {'PI (det)':<12} {'VI (det)':<12} {'PI (stoch)':<12} {'VI (stoch)':<12}")
for s in range(16):
    print(f"{s:<8} {results[False]['V_pi'][s]:<12.4f} {results[False]['V_vi'][s]:<12.4f} "
          f"{results[True]['V_pi'][s]:<12.4f} {results[True]['V_vi'][s]:<12.4f}")

# Policy comparison: argmax over actions per state
import numpy as np
action_names = ["←", "↓", "→", "↑"]
print("\nOptimal policy (deterministic | stochastic), reshaped as 4x4 grid:")
pi_det = np.argmax(results[False]['p_pi'], axis=1)
pi_sto = np.argmax(results[True]['p_pi'], axis=1)
for row in range(4):
    det_row = "  ".join(action_names[pi_det[row*4 + col]] for col in range(4))
    sto_row = "  ".join(action_names[pi_sto[row*4 + col]] for col in range(4))
    print(f"  {det_row}    |    {sto_row}")

Iterations to convergence:
                       deterministic   stochastic
Policy Iteration                   2            3
Value Iteration                    7          228

State values (all 16 states):
State    PI (det)     VI (det)     PI (stoch)   VI (stoch)  
0        0.9510       0.9510       0.5420       0.5420      
1        0.9606       0.9606       0.4988       0.4988      
2        0.9703       0.9703       0.4707       0.4707      
3        0.9606       0.9606       0.4568       0.4568      
4        0.9606       0.9606       0.5584       0.5584      
5        0.0000       0.0000       0.0000       0.0000      
6        0.9801       0.9801       0.3583       0.3583      
7        0.0000       0.0000       0.0000       0.0000      
8        0.9703       0.9703       0.5918       0.5918      
9        0.9801       0.9801       0.6431       0.6431      
10       0.9900       0.9900       0.6152       0.6152      
11       0.0000       0.0000       0.0000       0.0000      